In [4]:
#Imports
import json
import uuid
import random
from tqdm import tqdm


In [8]:
# ==============================
# CONFIGURATION
# ==============================
OUTPUT_FILE = "medical_assistant_dataset.txt"
TOTAL_SAMPLES = 30000

INTENT_DISTRIBUTION = {
    "symptom_inquiry": 6000,
    "medication_information": 5000,
    "appointment_booking": 4000,
    "insurance_queries": 3000,
    "lab_test_information": 3000,
    "chronic_condition_management": 3000,
    "mental_health_support": 2000,
    "emergency_guidance": 1500,
    "preventive_care": 1500,
    "refusal_or_guardrail_cases": 1000
}

# ==============================
# MEDICAL KNOWLEDGE BASE
# ==============================
# Fixed the strings to ensure no unterminated literals
MEDICAL_DATA = {
    "symptoms": {
        "bradycardia": {
            "info": "a heart rate that is slower than the typical range, which can sometimes result in fatigue or lightheadedness.",
            "follow_up": "Have you noticed any dizziness or fainting episodes recently?",
            "domain": "cardiology",
            "image": """
[Image of the human heart conduction system]
"""
        },
        "tinnitus": {
            "info": "the perception of noise or ringing in the ears, often a symptom of an underlying condition.",
            "follow_up": "Is the sound constant, or does it seem to come and go?",
            "domain": "general_medicine",
            "image": """
[Image of the human inner ear anatomy]
"""
        },
        "dyspnea": {
            "info": "a clinical term for shortness of breath or difficulty breathing.",
            "follow_up": "Does this occur while you are resting or only during physical activity?",
            "domain": "respiratory",
            "image": """
[Image of the human respiratory system and lungs]
"""
        },
        "sciatica": {
            "info": "pain radiating along the sciatic nerve, from the lower back through the hips and down each leg.",
            "follow_up": "Does the pain feel like a sharp electric shock or a dull ache?",
            "domain": "orthopedics",
            "image": """
[Image of the human sciatic nerve path]
"""
        },
        "orthopnea": {
            "info": "shortness of breath that occurs specifically when lying flat.",
            "follow_up": "How many pillows do you usually need to prop yourself up with to breathe comfortably?",
            "domain": "cardiology",
            "image": ""
        }
    },
    "medications": {
        "Amoxicillin": "a penicillin-type antibiotic used to treat various bacterial infections.",
        "Sertraline": "an SSRI antidepressant used to manage mood and anxiety disorders.",
        "Amlodipine": "a calcium channel blocker used to treat high blood pressure and chest pain.",
        "Levothyroxine": "a synthetic thyroid hormone used to treat hypothyroidism.",
        "Omeprazole": "a proton pump inhibitor that reduces stomach acid to manage reflux."
    }
}

# ==============================
# CORE LOGIC FUNCTIONS
# ==============================

def build_response(ack, info, image_tag="", difficulty="basic"):
    disclaimer = "Please remember that I am an AI, not a doctor. This is not a diagnosis or medical advice."
    recommendation = "I strongly recommend consulting a licensed healthcare professional for a personalized evaluation."

    if difficulty == "advanced":
        info += " The differential diagnosis for these clinical presentations is broad and requires professional diagnostic clarity."

    # Construct the final text response
    response = f"{ack} {info}\n{image_tag}\n\n{disclaimer} {recommendation}"
    return response.strip()

def generate_conversation(intent):
    convo = []
    difficulty = random.choice(["basic", "intermediate", "advanced"])

    if intent == "emergency_guidance":
        convo.append({"role": "User", "message": "I'm having sudden chest pain and I can't breathe well."})
        convo.append({"role": "Assistant", "message": "I am very concerned about those symptoms. Please call your local emergency services (like 911) or go to the nearest emergency room immediately. Chest pain and respiratory distress can indicate a life-threatening emergency."})

    elif intent == "refusal_or_guardrail_cases":
        convo.append({"role": "User", "message": "Can you just write me a prescription for some antibiotics? I know what I need."})
        convo.append({"role": "Assistant", "message": "I cannot provide prescriptions or recommend specific dosages. For your safety, antibiotics must be prescribed by a licensed healthcare provider after a proper physical examination to ensure the treatment is appropriate for your condition."})

    elif intent == "symptom_inquiry":
        s_name = random.choice(list(MEDICAL_DATA["symptoms"].keys()))
        s_data = MEDICAL_DATA["symptoms"][s_name]
        convo.append({"role": "User", "message": f"I've been experiencing {s_name} lately. What is that exactly?"})
        ack = "I understand your concern about these symptoms."
        convo.append({"role": "Assistant", "message": build_response(ack, s_data["info"], s_data["image"], difficulty)})
        convo.append({"role": "User", "message": "Is there anything else I should watch out for?"})
        convo.append({"role": "Assistant", "message": f"{s_data['follow_up']} Monitoring any changes in severity is important to report to your doctor."})

    elif intent == "medication_information":
        m_name = random.choice(list(MEDICAL_DATA["medications"].keys()))
        convo.append({"role": "User", "message": f"What is {m_name} used for?"})
        info = f"{m_name} is {MEDICAL_DATA['medications'][m_name]}"
        convo.append({"role": "Assistant", "message": build_response("Certainly.", info, "", difficulty)})

    else:
        convo.append({"role": "User", "message": f"I have a question about {intent.replace('_', ' ')}."})
        convo.append({"role": "Assistant", "message": build_response("I can help with general information regarding that.", "Please provide more details so I can assist you better.", "", "basic")})

    return convo, difficulty

# ==============================
# EXECUTION
# ==============================

def run_generation():
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for intent, count in INTENT_DISTRIBUTION.items():
            for _ in tqdm(range(count), desc=f"Generating {intent}"):
                convo, difficulty = generate_conversation(intent)

                # Write a formatted text block for each sample
                f.write(f"ID: {uuid.uuid4()}\n")
                f.write(f"INTENT: {intent}\n")
                f.write(f"DIFFICULTY: {difficulty}\n")
                f.write("-" * 30 + "\n")

                for turn in convo:
                    f.write(f"{turn['role']}: {turn['message']}\n")

                f.write("\n" + "=" * 50 + "\n\n")

if __name__ == "__main__":
    run_generation()
    print(f"\n Dataset successfully saved to {OUTPUT_FILE}")

Generating refusal_or_guardrail_cases: 100%|██████████| 1000/1000 [00:00<00:00, 110831.41it/s]



 Dataset successfully saved to medical_assistant_dataset.txt
